# Eager contraction on the 26-value grid

`contraction="eager"` (prototype-faithful: failed pattern moves also contract
the mesh), everything else identical to the head-to-head's new-algorithm run:
26-value grid, MAE objective, opportunistic poll, default 4-zone bullseye,
`random_state=0`. Question under test: does eager still find (4, 150, 17),
and in how many fits?

In [1]:
# --- Data pipeline: byte-for-byte replication of the prototype notebook ---
import time
import numpy as np
import pandas as pd

t0 = time.time()
trainBench = pd.read_csv(r"C:\FILES\Code\Benchmarking\train.csv")  # notebook: /home/dell/train.csv

SplitPoint = int(len(trainBench.index) * 0.8)
print("SplitPoint: ", SplitPoint)

df = trainBench

# int64 -> int32; object -> category -> numeric codes (Date included, as in the prototype)
Int64columns = df.select_dtypes(["int64"]).columns
df[Int64columns] = df[Int64columns].astype(np.int32)
cat_columns = df.select_dtypes(["object"]).columns
df[cat_columns] = df[cat_columns].astype("category")
df[cat_columns] = df[cat_columns].apply(lambda x: x.cat.codes)

# the five weather columns the prototype drops (note the dataset's own 'kM' typo)
df = df.drop("Max_Gust_SpeedKm_h", axis=1)
df = df.drop("CloudCover", axis=1)
df = df.drop("Max_VisibilityKm", axis=1)
df = df.drop("Min_VisibilitykM", axis=1)
df = df.drop("Mean_VisibilityKm", axis=1)

# positional 80/20 chronological split
trainBench, validBench = df.iloc[:SplitPoint, :], df.iloc[SplitPoint:, :]
print("Training Set shape", trainBench.shape)
print("Validation Set shape", validBench.shape)
del df

# feature mask: everything but the target (alphabetical order, incl. Date codes
# and NumberOfCustomers, exactly as the prototype had it)
mask = trainBench.columns.difference(["NumberOfSales"])
trainDataset_X = trainBench[mask]
trainDataset_y = trainBench["NumberOfSales"]
validBench_X = validBench[mask]
validBench_y = validBench["NumberOfSales"]
print("Feature Columns:")
print(list(mask))
del trainBench, validBench
print(f"Pipeline done in {time.time() - t0:.1f} s")

SplitPoint:  418416


C:\Users\Home\AppData\Local\Temp\ipykernel_2992\4255920482.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_columns = df.select_dtypes(["object"]).columns


Training Set shape (418416, 31)
Validation Set shape (104605, 31)
Feature Columns:
['AssortmentType', 'Date', 'Events', 'HasPromotions', 'IsHoliday', 'IsOpen', 'Max_Dew_PointC', 'Max_Humidity', 'Max_Sea_Level_PressurehPa', 'Max_TemperatureC', 'Max_Wind_SpeedKm_h', 'Mean_Dew_PointC', 'Mean_Humidity', 'Mean_Sea_Level_PressurehPa', 'Mean_TemperatureC', 'Mean_Wind_SpeedKm_h', 'Min_Dew_PointC', 'Min_Humidity', 'Min_Sea_Level_PressurehPa', 'Min_TemperatureC', 'NearestCompetitor', 'NumberOfCustomers', 'Precipitationmm', 'Region', 'Region_AreaKM2', 'Region_GDP', 'Region_PopulationK', 'StoreID', 'StoreType', 'WindDirDegrees']
Pipeline done in 4.5 s


In [2]:
# --- NEW PatternSearchCV, contraction="eager" — all else as in HeadToHead Run 1
import os
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from bayes_halving_search_cv import PatternSearchCV

X, y = trainDataset_X, trainDataset_y
N_features = X.shape[1]

def make_clf():
    return ExtraTreesRegressor(n_estimators=240, max_features=max(1, N_features - 15),
                               max_depth=16, n_jobs=1, random_state=0)

param_grid = {
    "max_features": [2, 3, 4],
    "n_estimators": list(range(10, 261, 10)),
    "max_depth": [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17],
}
tscv = TimeSeriesSplit(n_splits=5)

search = PatternSearchCV(make_clf(), param_grid,
                         scoring="neg_mean_absolute_error", cv=tscv,
                         n_jobs=-1, poll="opportunistic",
                         contraction="eager",
                         random_state=0, verbose=1)
t0 = time.time()
search.fit(X, y)
elapsed = time.time() - t0
print(f"EAGER search wall-clock: {elapsed:.2f} s")


[pattern_search_cv] subsample mode=expanding, zones=[0.1, 0.2, 0.5, 1.0] (rows [41842, 83684, 209208, 418416])


[pattern_search_cv] starts (1): [{'max_features': 3, 'n_estimators': 130, 'max_depth': 11}]


Fitting 5 folds for each of 1 candidates, totalling 5 fits


Fitting 5 folds for each of 1 candidates, totalling 5 fits


Fitting 5 folds for each of 1 candidates, totalling 5 fits


...

Fitting 5 folds for each of 1 candidates, totalling 5 fits


Fitting 5 folds for each of 1 candidates, totalling 5 fits


[pattern_search_cv] climber 0 converged at {'max_features': 4, 'n_estimators': 150, 'max_depth': 17} score=-805.7296842629627


[pattern_search_cv] engine done: 24 evaluations, 14 cache hits, climbers: {0: 'converged'}


EAGER search wall-clock: 688.78 s


In [3]:
res = search.cv_results_
n_fits = len(res["params"])
equiv = float(np.sum(np.asarray(res["n_resources"]) / len(y)))
best = search.best_params_
mae = -search.best_score_
cvres = cross_validate(make_clf().set_params(**best), X, y, cv=tscv,
                       scoring=("r2", "neg_mean_absolute_error"), n_jobs=-1)
r2 = cvres["test_r2"].mean()
print("=" * 72)
print("NEW PatternSearchCV, contraction='eager' - 26-value grid, MAE objective")
print("=" * 72)
print(f"evaluations          : {n_fits}")
print(f"full-fit equivalents : {equiv:.2f}")
print(f"wall-clock           : {elapsed:.1f} s")
print(f"best point           : ({best['max_features']}, {best['n_estimators']}, {best['max_depth']})")
print(f"CV MAE of best       : {mae:.3f}")
print(f"CV R2 of best        : {r2:.6f}")
print(f"zones used (rows)    : {sorted(set(int(v) for v in res['n_resources']))}")
print(f"cache hits           : {search.n_cache_hits_}")
print()
print("reference rows (same machine, same grid):")
print("  patient NEW      : 23 evals, 6.80 equiv, 824.1 s, (4,150,17), MAE 805.730")
print("  V1 prototype (R2): 33 evals, 33.00 equiv, 1710.9 s, (4,150,17), MAE 805.730")
print("  Optuna TPE       : 15 evals, 15.00 equiv, 828.7 s, (4,100,17), MAE 810.553")
print("  Optuna GP        : 15 evals, 15.00 equiv, 964.6 s, (4,150,17), MAE 805.730")
print()
print("evaluation-order trajectory:")
for p, s, nr in zip(res["params"], res["mean_test_score"], res["n_resources"]):
    print(f"  ({p['max_features']}, {p['n_estimators']:>3d}, {p['max_depth']:>2d})"
          f"  MAE {-s:9.3f}  rows {int(nr)}")


NEW PatternSearchCV, contraction='eager' - 26-value grid, MAE objective
evaluations          : 24
full-fit equivalents : 6.90
wall-clock           : 688.8 s
best point           : (4, 150, 17)
CV MAE of best       : 805.730
CV R2 of best        : 0.809981
zones used (rows)    : [41842, 418416]
cache hits           : 14

reference rows (same machine, same grid):
  patient NEW      : 23 evals, 6.80 equiv, 824.1 s, (4,150,17), MAE 805.730
  V1 prototype (R2): 33 evals, 33.00 equiv, 1710.9 s, (4,150,17), MAE 805.730
  Optuna TPE       : 15 evals, 15.00 equiv, 828.7 s, (4,100,17), MAE 810.553
  Optuna GP        : 15 evals, 15.00 equiv, 964.6 s, (4,150,17), MAE 805.730

evaluation-order trajectory:
  (3, 130, 11)  MAE  1028.629  rows 41842
  (4, 130, 11)  MAE   923.415  rows 41842
  (4, 250, 11)  MAE   944.135  rows 41842
  (4,  10, 11)  MAE  1010.617  rows 41842
  (4, 130, 17)  MAE   789.668  rows 41842
  (3, 130, 17)  MAE   878.465  rows 41842
  (4, 250, 17)  MAE   801.578  rows 41842
  (4